In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from keras.layers import GRU , Dense


In [ ]:
df = pd.read_excel("D:/Saham/penv/data.xlsx")
df = df.rename(columns={'Conclusion': 'y'})
df = df.drop(columns=['Company'] , axis=1)
df = df.replace('-', np.nan)
print(df.shape)
df.head()

In [ ]:
le = LabelEncoder()
# df['Industry'] = le.fit_transform(df['Industry'])
df.head()

In [ ]:
# convert all columns to float but Industry
cols_to_float = df.columns
df[cols_to_float] = df[cols_to_float].astype(float)
df.head()

In [ ]:
# drop rows which has no value in TSE_FY or RE_FY column
for i in ['TSE_FY', 'RE_FY']:
    for j in df.columns:
        if i in j :
            # print(j)
            df.dropna(subset=j)
print(df.shape)

In [ ]:
# print(df[df['y'] >= .5].shape)
# print(df[df['y'] < .5].shape)
# drop y
# df = df.drop(columns=['y'])

In [ ]:
# create a y column RE_FY / TSE_FY > .5 = 1 else 0
List_TSE_FY = ['TSE_FY_1390' ,
'TSE_FY_1391',
'TSE_FY_1392',
'TSE_FY_1393',
'TSE_FY_1394',
'TSE_FY_1395',
'TSE_FY_1396',
'TSE_FY_1397',
'TSE_FY_1398',
'TSE_FY_1399',
'TSE_FY_1400',
'TSE_FY_1401']
List_RE_FY = ['RE_FY_1390',
'RE_FY_1391',
'RE_FY_1392',
'RE_FY_1393',
'RE_FY_1394',
'RE_FY_1395',
'RE_FY_1396',
'RE_FY_1397',
'RE_FY_1398',
'RE_FY_1399',
'RE_FY_1400',
'RE_FY_1401']

mean_TSE_FY = df[List_TSE_FY].mean(axis=1).tolist()
df['mean_TSE_FY'] = mean_TSE_FY

mean_RE_FY = df[List_RE_FY].mean(axis=1).tolist()
df['mean_RE_FY'] = mean_RE_FY

df['y'] = np.where(df['mean_RE_FY'] / df['mean_TSE_FY'] > .5, 1, 0)
print(df[df['y'] >= .5].shape)
print(df[df['y'] < .5].shape)


In [ ]:
# print nan proportion
print(df.isna().sum() / df.shape[0])

In [ ]:
# df = df.fillna(df.mean())
# df = df.fillna(df.median())
# df = df.fillna(df.mode().iloc[0])

# fill NaN with 0
df = df.fillna(0)

print(df.isna().sum() / df.shape[0])

In [ ]:
# Scaler
scaler = MinMaxScaler()
df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
df.head()

In [ ]:
# split data
x = df.drop(columns=['y'])
y = df['y']
x = np.array(x)
y = np.array(y)
# [samples, timesteps, features]
x = x.reshape(x.shape[0], 1, x.shape[1])

x_train = x[:int(x.shape[0] * .7)]
y_train = y[:int(y.shape[0] * .7)]
x_valid = x[int(x.shape[0] * .7):int(x.shape[0] * .85)]
y_valid = y[int(y.shape[0] * .7):int(y.shape[0] * .85)]
x_test = x[int(x.shape[0] * .85):]
y_test = y[int(y.shape[0] * .85):]

print(x_train.shape)
print(y_train.shape)
print("------")
print(x_valid.shape)
print(y_valid.shape)
print("------")
print(x_test.shape)
print(y_test.shape)

In [ ]:
# duplicate data
DUPLICATE = 8
x_train = np.repeat(x_train, DUPLICATE, axis=0)
y_train = np.repeat(y_train, DUPLICATE, axis=0)
print(x_train.shape)
print(y_train.shape)

In [ ]:


rnn_model = keras.Sequential()
rnn_model.add(GRU(128, input_shape=(x_train.shape[1], x_train.shape[2])))
rnn_model.add(Dense(1, activation='sigmoid'))
rnn_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
rnn_model.summary()


In [ ]:
# train
history = rnn_model.fit(x_train, y_train, epochs=100, batch_size=64, validation_data=(x_valid, y_valid), shuffle=False)
# with no validation
# history = lstm.fit(x_train, y_train, epochs=100, batch_size=32, shuffle=False)

In [ ]:
# prin results
results = rnn_model.evaluate(x_test, y_test)
print(results)

In [ ]:
# plot loss
fig = go.Figure()
fig.add_trace(go.Scatter(y=history.history['loss'], mode='lines', name='loss'))
fig.add_trace(go.Scatter(y=history.history['val_loss'], mode='lines', name='val_loss'))
fig.update_layout(title='Loss', xaxis_title='Epoch', yaxis_title='Loss')
fig.show()

In [ ]:
# plot accuracy
fig = go.Figure()
fig.add_trace(go.Scatter(y=history.history['accuracy'], mode='lines', name='accuracy'))
fig.add_trace(go.Scatter(y=history.history['val_accuracy'], mode='lines', name='val_accuracy'))
fig.update_layout(title='Accuracy', xaxis_title='Epoch', yaxis_title='Accuracy')
fig.show()